# Scratch Studio Comments Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/22552/kasotest/blob/main/ScratchCommentsExplorer.ipynb)

`db.zst.part0` と `db.zst.part1` を結合して展開し、日本語の高速部分一致用 **FTS5 trigram** インデックスをローカルで作成して Gradio UI を起動します。

- 3文字以上: trigram で高速な部分一致
- 1〜2文字: LIKE に自動フォールバック
- ユーザー・期間・親/返信フィルタ
- 検索対象はDB全体、表示は1ページ200件
- 投稿数ランキング / 日別推移
- 読み取り専用SQL

> 圧縮DBは `--long=31` の zstd を2分割したものです。


In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!pip -q install "gradio>=6.20,<7" pandas plotly requests


In [ ]:
from pathlib import Path
import requests
import shutil
import sqlite3
import subprocess

PART_URLS = [
    "https://raw.githubusercontent.com/22552/kasotest/main/db.zst.part0",
    "https://raw.githubusercontent.com/22552/kasotest/main/db.zst.part1",
]
PART_PATHS = [Path("/content/db.zst.part0"), Path("/content/db.zst.part1")]
ZST_PATH = Path("/content/comments.sqlite.zst")
DB_PATH = Path("/content/comments.sqlite")

for url, path in zip(PART_URLS, PART_PATHS):
    tmp = Path(str(path) + ".tmp")
    tmp.unlink(missing_ok=True)
    print(f"[download] {path.name}")
    with requests.get(url, stream=True, timeout=(15, 180)) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length") or 0)
        done = 0
        with tmp.open("wb") as f:
            for chunk in r.iter_content(4 * 1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1048576:.1f}/{total/1048576:.1f} MiB ({done/total:.0%})", end="")
    print()
    tmp.replace(path)

print("[join] db.zst.part0 + db.zst.part1")
tmp_zst = Path(str(ZST_PATH) + ".tmp")
tmp_zst.unlink(missing_ok=True)
with tmp_zst.open("wb") as out:
    for path in PART_PATHS:
        with path.open("rb") as src:
            shutil.copyfileobj(src, out, 4 * 1024 * 1024)
tmp_zst.replace(ZST_PATH)

print("[zstd] decompress")
DB_PATH.unlink(missing_ok=True)
subprocess.run(["zstd", "-d", "--long=31", "-f", str(ZST_PATH), "-o", str(DB_PATH)], check=True)

with sqlite3.connect(DB_PATH) as con:
    has_trigram = con.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name='comments_trigram'"
    ).fetchone()
    if has_trigram:
        print("[index] trigram index is bundled in DB")
    else:
        print("[index] build trigram substring index once")
        con.execute("PRAGMA journal_mode=OFF")
        con.execute("PRAGMA synchronous=OFF")
        con.execute("""CREATE VIRTUAL TABLE comments_trigram USING fts5(
            content,
            content='comments',
            content_rowid='id',
            tokenize='trigram'
        )""")
        con.execute("INSERT INTO comments_trigram(comments_trigram) VALUES('rebuild')")
        con.commit()

print(f"[ready] {DB_PATH} ({DB_PATH.stat().st_size/1048576:.1f} MiB)")


In [ ]:
import math
import sqlite3
import pandas as pd
import plotly.express as px
import gradio as gr

PAGE_SIZE = 200

def db_connect():
    con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, timeout=5, check_same_thread=False)
    con.execute("PRAGMA query_only=ON")
    con.execute("PRAGMA busy_timeout=5000")
    return con

def build_search(query, mode, user, start, end, kind):
    clauses, params = [], []

    if kind == "親コメント":
        clauses.append("c.is_reply=0")
    elif kind == "返信":
        clauses.append("c.is_reply=1")

    user = (user or "").strip()
    if user:
        clauses.append("c.user=? COLLATE NOCASE")
        params.append(user)

    start = (start or "").strip()
    if start:
        clauses.append("c.datetime>=?")
        params.append(start)

    end = (end or "").strip()
    if end:
        clauses.append("c.datetime<=?")
        params.append(end + ("T23:59:59.999999Z" if len(end) == 10 else ""))

    q = (query or "").strip()

    if q and mode == "FTS5（単語一致）":
        where = " AND ".join(["comments_fts MATCH ?"] + clauses)
        return (
            """SELECT c.id,c.parent_id,c.is_reply,c.user,c.user_id,c.datetime,c.content,
                      bm25(comments_fts) AS score """,
            f"""FROM comments_fts
                JOIN comments c ON c.id=comments_fts.rowid
                WHERE {where}""",
            " ORDER BY score,c.datetime DESC",
            [q] + params,
        )

    if q and len(q) >= 3 and mode == "自動（高速部分一致）":
        literal = '"' + q.replace('"', '""') + '"'
        where = " AND ".join(["comments_trigram MATCH ?"] + clauses)
        return (
            "SELECT c.id,c.parent_id,c.is_reply,c.user,c.user_id,c.datetime,c.content ",
            f"""FROM comments_trigram
                JOIN comments c ON c.id=comments_trigram.rowid
                WHERE {where}""",
            " ORDER BY c.datetime DESC",
            [literal] + params,
        )

    if q:
        clauses.insert(0, "c.content LIKE ?")
        params.insert(0, f"%{q}%")

    where = (" WHERE " + " AND ".join(clauses)) if clauses else ""
    return (
        "SELECT c.id,c.parent_id,c.is_reply,c.user,c.user_id,c.datetime,c.content ",
        "FROM comments c" + where,
        " ORDER BY c.datetime DESC",
        params,
    )

def search_page(query, mode, user, start, end, kind, page):
    try:
        page = max(1, int(page or 1))
    except Exception:
        page = 1

    try:
        select, base, order, params = build_search(query, mode, user, start, end, kind)
        with db_connect() as con:
            total = con.execute("SELECT COUNT(*) " + base, params).fetchone()[0]
            pages = max(1, math.ceil(total / PAGE_SIZE))
            page = min(page, pages)
            offset = (page - 1) * PAGE_SIZE
            df = pd.read_sql_query(
                select + base + order + " LIMIT ? OFFSET ?",
                con,
                params=params + [PAGE_SIZE, offset],
            )
    except Exception as e:
        return pd.DataFrame(), f"❌ {type(e).__name__}: {e}", 1

    if not df.empty:
        df["種別"] = df["is_reply"].map({0: "親", 1: "返信"})
        cols = ["id", "parent_id", "種別", "user", "user_id", "datetime", "content"]
        if "score" in df.columns:
            cols.append("score")
        df = df[cols]

    first = 0 if total == 0 else offset + 1
    last = min(offset + len(df), total)
    used = "trigram" if (query or "").strip() and len((query or "").strip()) >= 3 and mode == "自動（高速部分一致）" else mode
    status = f"✅ **{total:,}件ヒット** — {first:,}〜{last:,}件 / **{page:,}/{pages:,}ページ**　`{used}`"
    return df, status, page

def stats():
    with db_connect() as con:
        total, top, replies, users, old, new = con.execute(
            """SELECT COUNT(*),SUM(is_reply=0),SUM(is_reply=1),
                      COUNT(DISTINCT user),MIN(datetime),MAX(datetime)
               FROM comments"""
        ).fetchone()
    return f"""### DB概要
- 全行数: **{total:,}**
- 親コメント: **{top:,}**
- 返信: **{replies:,}**
- ユーザー数: **{users:,}**
- 期間: `{old}` ～ `{new}`"""

def top_users(n):
    with db_connect() as con:
        return pd.read_sql_query(
            "SELECT user,COUNT(*) comments FROM comments GROUP BY user ORDER BY comments DESC LIMIT ?",
            con, params=[int(n)]
        )

def daily_plot():
    with db_connect() as con:
        df = pd.read_sql_query(
            "SELECT substr(datetime,1,10) day,COUNT(*) comments FROM comments GROUP BY day ORDER BY day",
            con
        )
    return px.line(df, x="day", y="comments", title="日別コメント数")

def run_sql(sql):
    text = (sql or "").strip().rstrip(";").strip()
    if not text:
        return pd.DataFrame(), "SQLを入力してください"
    if ";" in text:
        return pd.DataFrame(), "❌ 複数文は禁止"
    if text.split(None, 1)[0].upper() not in {"SELECT", "WITH", "EXPLAIN"}:
        return pd.DataFrame(), "❌ 読み取り専用です"
    try:
        with db_connect() as con:
            cur = con.execute(text)
            cols = [d[0] for d in (cur.description or [])]
            rows = cur.fetchmany(1001)
        clipped = len(rows) > 1000
        rows = rows[:1000]
        return pd.DataFrame(rows, columns=cols), f"✅ {len(rows)}行" + ("（先頭1000行）" if clipped else "")
    except Exception as e:
        return pd.DataFrame(), f"❌ {type(e).__name__}: {e}"

with gr.Blocks(title="Scratch Studio Comments Explorer") as demo:
    gr.Markdown("# 🔎 Scratch Studio Comments Explorer\n3文字以上の検索は **trigram** で高速部分一致します。1〜2文字だけLIKEにフォールバックします。")

    with gr.Tab("検索"):
        with gr.Row():
            q = gr.Textbox(label="本文検索")
            mode = gr.Dropdown(
                ["自動（高速部分一致）", "FTS5（単語一致）", "LIKE（低速）"],
                value="自動（高速部分一致）",
                label="方式",
            )
        with gr.Row():
            user = gr.Textbox(label="ユーザー")
            kind = gr.Dropdown(["すべて", "親コメント", "返信"], value="すべて", label="種別")
        with gr.Row():
            start = gr.Textbox(label="開始", placeholder="2023-09-05")
            end = gr.Textbox(label="終了")

        search_btn = gr.Button("検索", variant="primary")
        status = gr.Markdown()
        table = gr.Dataframe()

        with gr.Row():
            prev_btn = gr.Button("← 前の200件")
            page = gr.Number(value=1, minimum=1, precision=0, label="ページ")
            jump_btn = gr.Button("ページへ移動")
            next_btn = gr.Button("次の200件 →")

        inputs = [q, mode, user, start, end, kind]
        search_btn.click(lambda *x: search_page(*x, 1), inputs, [table, status, page])
        prev_btn.click(lambda *x: search_page(*x[:-1], max(1, int(x[-1] or 1) - 1)), inputs + [page], [table, status, page])
        next_btn.click(lambda *x: search_page(*x[:-1], int(x[-1] or 1) + 1), inputs + [page], [table, status, page])
        jump_btn.click(search_page, inputs + [page], [table, status, page])

    with gr.Tab("統計"):
        s = gr.Markdown()
        gr.Button("概要").click(stats, outputs=s)
        n = gr.Slider(10, 200, 50, step=10, label="上位ユーザー数")
        ut = gr.Dataframe()
        gr.Button("ランキング").click(top_users, n, ut)
        plot = gr.Plot()
        gr.Button("日別グラフ").click(daily_plot, outputs=plot)

    with gr.Tab("SQL"):
        sql = gr.Code(
            value="SELECT user, COUNT(*) AS comments FROM comments GROUP BY user ORDER BY comments DESC LIMIT 100;",
            language="sql",
            label="SQL",
        )
        msg = gr.Markdown()
        out = gr.Dataframe()
        gr.Button("実行", variant="primary").click(run_sql, sql, [out, msg])

demo.queue(default_concurrency_limit=2).launch(share=True, debug=False)
